# BDC Yield — a quantitative teardown 🔬
### Real total-return tapes · yield-vs-realised gap · full vs downside beta · HAC + block-bootstrap · crash co-movement

![Signal: Real](https://img.shields.io/badge/Signal-Real-2ea44f?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Survives credit cycles?: Busted](https://img.shields.io/badge/Survives_credit_cycles%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* We test the three things the BDC label sells — a **~10% yield** (only ~6% of it survives as total return), **bond-like steadiness** (mirage), and **income that survives credit cycles** (busted) — and pin down *what BIZD actually is* via its beta to stocks vs bonds, in the body and in the tail.

> ⚠️ **Not investment advice.** BIZD/SPY/IEF daily, **total-return** adjusted (`bdc_yield.data`, yfinance `auto_adjust=True`); betas via OLS with a HAC *t* and a circular block bootstrap on the difference. BIZD inception bounds the window at 2013-02-12. Sources in [`docs/references.md`](../docs/references.md), reproducible run in [`docs/results.md`](../docs/results.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # study root (bdc_yield/)
sys.path.insert(0, os.path.abspath("../../.."))    # repo root (quantlab/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from bdc_yield import data, strategy

AS_OF = "2026-05-31"
QUOTED_YIELD = 0.10   # BIZD's headline distribution rate, ~10%
# Try the REAL total-return tapes; fall back to the offline synthetic control if the
# network is blocked. Every cell prints which TAPE it is showing.
try:
    prices = data.load_real(("BIZD", "SPY", "IEF")).loc[:AS_OF]
    cols = {"BIZD": "BIZD", "SPY": "SPY", "BND": "IEF"}   # bond proxy label
    TAPE = "REAL"
except Exception as e:
    print("network/cache miss -> SYNTHETIC control tape:", type(e).__name__)
    prices, _truth = data.synthetic_three_asset(bdc_beta=1.15, seed=342)
    cols = {"BIZD": "BIZD", "SPY": "SPY", "BND": "BND"}
    TAPE = "SYNTHETIC"

BIZD, SPY, BND = cols["BIZD"], cols["SPY"], cols["BND"]
rets = strategy.to_returns(prices)
BANNER = ("REAL total-return tape (BIZD/SPY/IEF, 2013-2026)" if TAPE == "REAL"
          else "SYNTHETIC control tape (bdc_beta=1.15) -- NOT market data")
print(f"[{TAPE}] {BANNER}")
print(f"panel: {len(prices):,} rows  {prices.index[0].date()} -> {prices.index[-1].date()}  fingerprint={data.fingerprint(prices)}")
for c in (BIZD, SPY, BND):
    s = strategy.stats(rets[c])
    print(f"  {c}: CAGR {s['cagr']*100:6.2f}%  vol {s['vol']*100:5.1f}%  Sharpe {s['sharpe']:.3f}  maxDD {s['max_dd']*100:6.1f}%")


[REAL] REAL total-return tape (BIZD/SPY/IEF, 2013-2026)
panel: 3,344 rows  2013-02-12 -> 2026-05-29  fingerprint=1773bea09c5d
  BIZD: CAGR   6.21%  vol  20.1%  Sharpe 0.401  maxDD  -55.4%
  SPY: CAGR  14.81%  vol  16.9%  Sharpe 0.903  maxDD  -33.7%
  IEF: CAGR   1.32%  vol   6.5%  Sharpe 0.235  maxDD  -23.9%


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| **Signal** | `REAL` | Beta of BIZD to **stocks** **+0.722** (HAC *t* **+5.48**), to **bonds** -0.241; downside beta to stocks **1.1** (>1); fell in 6/6 crashes. |
| **Tradability** | `MIRAGE` | Only **6.21%** of the **~10%** headline survived (CAGR < SPY's 14.81%) for vol **20.1%** (>SPY) and maxDD **-55.4%**. Not a safe income sleeve. |
| **Survives credit cycles?** | `BUSTED` | In COVID BIZD lost -55.0% (worse than SPY) while bonds gained +6.4%. A distribution rate ≠ bond income. |

> 💡 **In plain words:** BIZD is **levered private-credit equity wearing a yield label**. The risk is statistically real; the safe high yield is the part that isn't there.

## 1 · The claim, steelmanned

- **H₁ (yield):** the ~10% distribution arrives as ~10% of *total return*.
- **H₂ (safety):** BIZD's risk (vol, drawdown, crash co-movement) is bond-like, not (levered-)equity-like.
- **H₃ (survives the cycle):** BIZD holds up — or at least doesn't fall harder than stocks — when credit/equity crashes.

H₁ is **false** (only ~6.2% of 10% survives), and H₂/H₃ are **false** — BIZD loads on equities at beta >0.7, *negatively* on bonds, and the loading exceeds 1 in the tail.

## 2 · So what? — what rides on each answer

If H₂ holds, BIZD belongs in the *income* sleeve and the 2020 result is an aberration. If BIZD is levered-equity-in-disguise, every income investor filing it under fixed income is badly mis-stating their true equity exposure — and discovers it in the next recession, when the credit cuts and the price cut arrive together. Same 'is this asset what the brochure says?' question the desk asked of preferreds ([Study 338](../../338-preferred-stocks/)) and bank loans ([Study 340](../../340-bank-loans/)).

## 3 · How we'd know — the protocol

Yield-vs-realised total-return gap · return/vol/drawdown on the total-return tape · full-sample beta of BIZD to SPY and to IEF, each with a **HAC *t*** on the OLS-slope influence series · **downside** beta on the worst 10% of equity days · crash co-movement in every >10% equity drawdown · a **circular block bootstrap** CI on the downside-beta *difference* (to-stocks − to-bonds).

## 4 · The teardown

First, the yield illusion — the ~10% headline vs what arrived:

In [2]:
bizd_cagr = strategy.stats(rets[BIZD])['cagr']
hv = strategy.headline_vs_realised(bizd_cagr, QUOTED_YIELD)
print(f'[{TAPE}] quoted distribution yield  = {hv["quoted_yield"]*100:6.1f}%')
print(f'[{TAPE}] realised total-return CAGR = {hv["realised_cagr"]*100:6.2f}%')
print(f'[{TAPE}] phantom gap = {hv["gap"]*100:6.2f} pts  ({hv["fraction_phantom"]*100:.0f}%)')

[REAL] quoted distribution yield  =   10.0%
[REAL] realised total-return CAGR =   6.21%
[REAL] phantom gap =   3.79 pts  (38%)


Headline stats — the return is below stocks, the risk is above them:

In [3]:
tbl = pd.DataFrame({
  'CAGR %':  [strategy.stats(rets[c])['cagr']*100 for c in (BIZD, SPY, BND)],
  'Vol %':   [strategy.stats(rets[c])['vol']*100 for c in (BIZD, SPY, BND)],
  'Sharpe':  [strategy.stats(rets[c])['sharpe'] for c in (BIZD, SPY, BND)],
  'MaxDD %': [strategy.stats(rets[c])['max_dd']*100 for c in (BIZD, SPY, BND)],
}, index=[BIZD, f'{SPY} (equity)', f'{BND} (bonds)'])
print(f'[{TAPE}] tape')
tbl.round(2)

[REAL] tape


,CAGR %,Vol %,Sharpe,MaxDD %
BIZD,6.210,20.090,0.400,-55.440
SPY (equity),14.810,16.890,0.900,-33.720
IEF (bonds),1.320,6.460,0.240,-23.920


**Who does BIZD move with?** OLS beta to stocks and to bonds, each with a HAC *t* on the slope's influence series (mean of the influence series = the OLS slope).

In [4]:
def hac_beta_t(p, x):
    p = p.to_numpy(float); x = x.to_numpy(float); xc = x - x.mean()
    infl = (p - p.mean()) * xc / (xc @ xc) * len(xc)
    return float(infl.mean()), strategy.hac_tstat(infl)
b_spy, t_spy = hac_beta_t(rets[BIZD], rets[SPY])
b_bnd, t_bnd = hac_beta_t(rets[BIZD], rets[BND])
print(f'[{TAPE}]  beta BIZD~{SPY} = {b_spy:+.3f}  (HAC t {t_spy:+.2f})')
print(f'[{TAPE}]  beta BIZD~{BND} = {b_bnd:+.3f}  (HAC t {t_bnd:+.2f})')
print(f'downside beta to {SPY} (worst 10% days) = {strategy.downside_beta(rets[BIZD], rets[SPY]):.3f}')
print(f'downside beta to {BND} (worst 10% days) = {strategy.downside_beta(rets[BIZD], rets[BND]):.3f}')

[REAL]  beta BIZD~SPY = +0.722  (HAC t +5.48)
[REAL]  beta BIZD~IEF = -0.241  (HAC t -2.45)
downside beta to SPY (worst 10% days) = 1.098


downside beta to IEF (worst 10% days) = 2.277


> 💡 **In plain words:** on the real tape the beta to stocks is **+0.722** with a HAC *t* of **+5.48** — far past the *t*=2 bar — while the beta to bonds is *negative*. And on the worst equity days the stock-beta climbs **above 1** (~1.1): BIZD inherits *more* than the full equity move precisely when 'income safety' was the point. That is leverage, not a cushion.

**Block-bootstrap on the difference** — is BIZD's downside beta to stocks reliably above its downside beta to bonds? (Tail-conditioned betas are noisy — bonds barely move on the worst equity days, so the to-bonds tail beta is unstable — so we resample jointly in blocks and report the CI honestly.)

In [5]:
boot = strategy.bootstrap_downside_beta_diff(rets[BIZD], rets[SPY], rets[BND], block=21, n_boot=2000, seed=342)
print(f'[{TAPE}] downside-beta diff (to-{SPY} minus to-{BND}) = {boot["point"]:+.3f}')
print(f'  95% block-bootstrap CI [{boot["ci95"][0]:+.3f}, {boot["ci95"][1]:+.3f}]')
print(f'  BIZD more equity-like in {boot["frac_spy_wins"]*100:.0f}% of resamples')

[REAL] downside-beta diff (to-SPY minus to-IEF) = -1.179
  95% block-bootstrap CI [-2.923, +0.562]
  BIZD more equity-like in 15% of resamples


> 💡 **In plain words:** the **certified** result rests on the full-sample beta (HAC *t* +5.48, far past 2) and the >1 downside beta to stocks. The tail-beta *difference* bootstrap is deliberately shown warts-and-all: because Treasuries barely move on the worst equity days, the to-bonds tail beta is numerically unstable and the difference CI is wide — so we do **not** lean on it for the stamp. The stamp comes from the body-of-the-distribution beta, which is unambiguous.

**Crash co-movement** — every >10% equity drawdown, BIZD and bonds over the same window:

In [6]:
eps = strategy.equity_drawdowns(rets[[SPY, BIZD, BND]], SPY, thresh=-0.10)
rows = [{'peak':e['peak'].date(),'trough':e['trough'].date(),
         f'{SPY} %':e['stock_loss']*100, f'{BIZD} %':e['others'][BIZD]*100, f'{BND} %':e['others'][BND]*100} for e in eps]
dd = pd.DataFrame(rows)
print(f'[{TAPE}] BIZD fell in {(dd[f"{BIZD} %"]<0).sum()}/{len(dd)}; bonds rose in {(dd[f"{BND} %"]>0).sum()}/{len(dd)}')
dd.round(1)

[REAL] BIZD fell in 6/6; bonds rose in 4/6


,peak,trough,SPY %,BIZD %,IEF %
0,2015-07-20,2016-02-11,-13.000,-18.100,7.400
1,2018-01-26,2018-02-08,-10.100,-5.000,-1.200
2,2018-09-20,2018-12-24,-19.300,-15.400,3.400
3,2020-02-19,2020-03-23,-33.700,-55.000,6.400
4,2022-01-03,2022-10-12,-24.500,-15.800,-15.400
5,2025-02-19,2025-04-08,-18.800,-19.600,2.700


## 5 · The verdict

Signal `REAL` (beta to stocks +0.722, HAC *t* +5.48; downside beta >1; fell 6/6 crashes). Tradability `MIRAGE` (only 6.21% of the ~10% survived, < SPY's 14.81%, for vol 20.1% > SPY and maxDD -55.4%). Survives-credit-cycles? `BUSTED` (COVID BIZD -55.0% vs bonds +6.4%).

## 6 · Could you trade it?

Capacity and liquidity are fine in calm markets — BIZD trades a healthy volume. The reservations are structural, not frictional: (1) BDCs are **externally managed with fat fees and balance-sheet leverage** (regulatory debt/equity up to ~2:1 since the 2018 SBCAA), so the fund equity amplifies the loan book; (2) the borrowers are **private, sub-investment-grade, often PE-owned** mid-market firms — the credits that default first in a recession; (3) the ~10% is compensation for that risk, not a free yield premium. As a high-yield *equity* understood as risk: defensible. As the *safe income* sleeve: a mirage.

## 7 · Going further

- Decompose the distribution into **net investment income vs return of capital** (N-CSR/19a-1 filings): how much of the ~10% is genuinely earned?
- Re-run on **individual large BDCs** (ARCC, MAIN, FSK) — does an internally-managed, lower-fee BDC escape the COVID result, or is the leverage structural?
- Add the **discount/premium to NAV** as a regime variable: BDCs swing to deep discounts in stress, the price-vs-NAV gap that turns a credit wobble into a 55% drawdown.